# 2. Patch Extraction
Reads DICOM whole-slide images, maps annotations to tiles,
and extracts 64x64 patches into mitotic/ and non_mitotic/ directories.
Uses multiprocessing to process slides in parallel.

In [ ]:
import pandas as pd
import multiprocessing as mp
from pathlib import Path
from wsi_utils import process_slide

In [ ]:
base_dir = Path("D:/Mitosis WSI CCMCT")

In [ ]:
# Load and merge annotation tables
df_annots = pd.read_csv(base_dir / "meta_data" / "Annotations.csv")
df_coords = pd.read_csv(base_dir / "meta_data" / "Annotations_coordinates.csv")
df = pd.merge(df_annots, df_coords, on="uid")

# Build slide_id -> DICOM path mapping
df_slides = pd.read_csv(base_dir / "meta_data" / "Slides.csv")
slide_to_path = {
    row["uid"]: base_dir / "training_data" / row["filename"]
    for _, row in df_slides.iterrows()
}

# Group annotations by slide
grouped_list = [(slide_id, group.copy()) for slide_id, group in df.groupby("slide_x")]

In [ ]:
mp.set_start_method("spawn", force=True)

if __name__ == "__main__":
    with mp.Pool(processes=6) as pool:
        pool.starmap(
            process_slide,
            [(sid, grp, slide_to_path, base_dir) for sid, grp in grouped_list]
        )
    print("Extraction complete!")